# Quickstart

Walk through the **gsea_squared** pipeline ((Balanis et al. 2019) on a toy GSEA dataset:
preprocess pathway names, assign categories via regex, run KS enrichment, and plot results.
This is the same regex → enrichment chain that `scripts/gsea_squared.py` runs end-to-end.

**Runtime:** < 2 min on Colab CPU (no GPU needed).

## Setup

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !git clone https://github.com/fesedebe/longtrac-tumor-evolution-framework.git repo
    os.chdir("repo")
    !pip install -q -e .
else:
    # Running locally from notebooks/ — move to repo root
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from gsea_refiner.preprocessing.clean import clean_gene_set_name
from gsea_refiner.labeling.regex import label_pathways_by_regex
from gsea_refiner.enrichment.ks import run_category_enrichment

## 1. Load toy GSEA data

In [ ]:
df = pd.read_csv("data/example/gsea_toy.tsv", sep="\t")
print(f"{len(df)} pathways")
df.head()

## 2. Preprocess pathway names

In [ ]:
df["cleaned"] = df["pathway"].apply(clean_gene_set_name)
df[["pathway", "cleaned"]].head(8)

## 3. Assign categories with regex labeling

Load the category keyword patterns and apply regex matching to the raw pathway names.

In [ ]:
kw = pd.read_csv("data/config/category_keywords.csv")
categories = kw["Category"].tolist()
patterns = kw["Regex"].tolist()
print(f"Categories: {categories}")

df_labeled = label_pathways_by_regex(
    df, categories=categories, cat_terms=patterns, col="pathway", label_col="Category"
)
df_labeled[["pathway", "NES", "Category"]].head(10)

In [ ]:
df_labeled["Category"].value_counts()

## 4. Run KS enrichment per category

Test whether each category's pathways cluster at one end of the NES-ranked list.

In [ ]:
results = run_category_enrichment(
    df_labeled,
    categories=categories,
    prediction_col="Category",
    verbose=True,
)

cat_stats = results["categories"]
cat_stats

## 5. Plot enrichment results

In [ ]:
cat_stats_plot = cat_stats[cat_stats["Freq"] > 0].sort_values("signedlogp")

colors = ["#d73027" if v > 0 else "#4575b4" for v in cat_stats_plot["signedlogp"]]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(cat_stats_plot["Category"], cat_stats_plot["signedlogp"], color=colors)
ax.set_xlabel("Signed log10(p-value)")
ax.set_title("Category enrichment on toy GSEA data")
ax.axvline(0, color="grey", linewidth=0.5)
for i, (freq, pval) in enumerate(zip(cat_stats_plot["Freq"], cat_stats_plot["pval"])):
    ax.annotate(f"n={freq}, p={pval:.2e}", xy=(0.01, i), fontsize=8, va="center")
plt.tight_layout()
plt.show()

## 6. Inspect ranked pathways

In [ ]:
ranked = results["pathways"].sort_values("NES", ascending=False)
ranked[["pathway", "NES", "padj", "Category", "rank"]]

---

**Next steps:** Replace the regex labeling (step 3) with a fine-tuned BioBERT classifier for better generalization. See `scripts/02_train_classifier.py` and `scripts/03_predict_categories.py`.